In [1]:
#Currently supports 1 race at a time, will implement vectorized environments later
import pystk2
import multiprocessing as mp

from collections import deque
import numpy as np

import torch
import torch.optim as optim
import torch.nn.functional as F
from actor import ActorNetwork
from critic import CriticNetwork
#import os
#os.add_dll_directory(r'C:\Program Files\SuperTuxKart 1.5')

In [2]:
class ProcessState:
	def __init__(self, max_speed=30, map_size=100, track_length=2000):
		self.frame = deque(maxlen=4)    # Window holding last 4 frames
		self.max_speed = max_speed
		self.map_size = map_size
		self.track_length = track_length

	def processObservation(self, obs):
		loc = np.array(obs["location"], dtype=np.float32) / self.map_size
		loc = np.clip(loc, -1.0, 1.0)

		vel = np.array(obs["velocity"], dtype=np.float32) / self.max_speed
		vel = np.clip(vel, -1.0, 1.0)

		front = np.array(obs["front"], dtype=np.float32) / self.map_size
		front = np.clip(front, -1.0, 1.0)

		jump = np.array([1.0 if obs["jumping"] else 0.0], dtype=np.float32)

		rotation = np.array(obs["rotation"], dtype=np.float32)

		dist = np.array([obs.get("distance_down_track", 0.0)], dtype=np.float32) / self.track_length

		state = np.concatenate([loc, vel, front, jump, rotation, dist])

		if len(self.frame) == 0:
			for _ in range(4):
				self.frame.append(state)
		else:
			self.frame.append(state)

		return np.concatenate(self.frame)

In [3]:
# --- STEP 1: THE PPO UPDATE FUNCTION ---

# Instantiate models with an input dimension of 60
actor_net = ActorNetwork(state_dim=60)
critic_net = CriticNetwork(state_dim=60)

# Set to evaluation mode for simulation
actor_net.eval()
critic_net.eval()

# Initialize Optimizer for both networks
optimizer = optim.Adam([
	{'params': actor_net.parameters(), 'lr': 3e-4},
	{'params': critic_net.parameters(), 'lr': 3e-4}
])

def update_ppo(buffer, epochs=4, gamma=0.99, clip_epsilon=0.2):
	# Extract Data from Buffer
	states = torch.FloatTensor([t['state'] for t in buffer])
	
	# Reconstruct the original Actor format of [Steering, Acceleration].
	actions = torch.FloatTensor([[t['action'][1], t['action'][0]] for t in buffer])
	
	rewards = [t['reward'] for t in buffer]
	values = torch.FloatTensor([t['value'] for t in buffer]).unsqueeze(1)
	old_log_probs = torch.FloatTensor([t['log_prob'] for t in buffer]).unsqueeze(1)
	
	# Calculate Discounted Returns
	returns = []
	discounted_reward = 0
	for reward in reversed(rewards):
		discounted_reward = reward + (gamma * discounted_reward)
		returns.insert(0, discounted_reward)
	returns = torch.FloatTensor(returns).unsqueeze(1)
	
	# Calculate Advantages
	advantages = returns - values
	# Added 1e-8 to prevent division by zero
	advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)
	
	# Switch models back to train mode
	actor_net.train()
	critic_net.train()
	
	# PPO Update Loop
	for _ in range(epochs):
		# Check for corrupted engine data
		if torch.isnan(states).any():
			print("NaN detected in states buffer! Skipping PPO update.")
			break
			
		action_dists = actor_net(states)
		new_values = critic_net(states)
		
		new_log_probs = action_dists.log_prob(actions).sum(dim=-1, keepdim=True)
		entropy = action_dists.entropy().sum(dim=-1, keepdim=True).mean()
		
		ratio = torch.exp(new_log_probs - old_log_probs)
		
		surr1 = ratio * advantages
		surr2 = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * advantages
		
		actor_loss = -torch.min(surr1, surr2).mean()
		critic_loss = F.mse_loss(new_values, returns)
		
		loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy
		
		optimizer.zero_grad()
		loss.backward()
		
		# Gradient Clipping to prevent exploding weights
		torch.nn.utils.clip_grad_norm_(actor_net.parameters(), max_norm=0.5)
		torch.nn.utils.clip_grad_norm_(critic_net.parameters(), max_norm=0.5)
		
		optimizer.step()
		
	actor_net.eval()
	critic_net.eval()

In [4]:
def SingleInstance(rank,pipe):
	pystk2.init(pystk2.GraphicsConfig.none())
	WorldState = pystk2.WorldState()
	config = pystk2.RaceConfig(track='lighthouse', num_kart=1, laps=1)
	config.players[0].controller = pystk2.PlayerConfig.Controller.PLAYER_CONTROL
	race = pystk2.Race(config)
	try:
		race.start()
			
		# Track details must be loaded after race start
		track = pystk2.Track()
		track.update()
		track_length = track.length
		max_coordinate = np.max(np.abs(track.path_nodes))
		
		processor = ProcessState(max_speed=30, map_size=max_coordinate, track_length=track_length)
		RaceEnded = False
		reward = 0.0

		WorldState.update()
			
		kart = WorldState.karts[0]
		obs = {
			"location": kart.location,
			"velocity": kart.velocity,
			"front": kart.front,
			"jumping": kart.jumping,
			"rotation": kart.rotation,
			"distance_down_track": kart.distance_down_track
		}
		np_obs = processor.processObservation(obs=obs)
		#Send observation to model
		pipe.send([np_obs,reward,RaceEnded])

		while True:
			#Receive action from model
			ActionMessage = pipe.recv()

			if ActionMessage == 'TERMINATE':
				return

			action = pystk2.Action()
			action.steer = ActionMessage[0]
			action.acceleration = ActionMessage[1]

			# Step the environment
			RaceEnded = not race.step(action)

			WorldState.update()

			kart = WorldState.karts[0]
			obs = {
				"location": kart.location,
				"velocity": kart.velocity,
				"front": kart.front,
				"jumping": kart.jumping,
				"rotation": kart.rotation,
				"distance_down_track": kart.distance_down_track
			}
			np_obs = processor.processObservation(obs=obs)

			# Reward Calculation
			vel_x, vel_y, vel_z = obs['velocity']
			speed = (vel_x**2 + vel_y**2 + vel_z**2)**0.5
			reward = speed * 0.1
			
			if obs.get('distance_down_track', 0.0) < 0:
				reward -= 10.0
				
			#Send observation to model
			pipe.send([np_obs,reward,RaceEnded])

	finally:
		# Critical Cleanup
		race.stop()
		del race
		pystk2.clean()

In [5]:
def main():
	try:
		PATIENCE_LIMIT = 50
		best_reward = float('-inf')
		patience_counter = 0

		for episode in range(100):
			buffer = []
			total_episode_reward = 0.0
			ProcessList = []
			ConList = []
			for i in range(5):
				ParentCon,ChildCon = mp.Pipe()
				process = mp.Process(target=SingleInstance,args=(i,ChildCon))
				ProcessList.append(process)
				ConList.append(ParentCon)
				process.start()
			BatchStates = []
			BatchDones = []

			for con in ConList:
					np_obs,reward,RaceDone = con.recv()
					BatchStates.append(np_obs)
					BatchDones.append(RaceDone)

			for step in range(1000):

				# --- Phase 2 Brain Injection ---
				state_tensor = torch.FloatTensor(np.array(BatchStates))
				
				# Sanity Check for incoming engine observations
				if torch.isnan(state_tensor).any():
					print("NaN detected in engine observations! Terminating episode.")
					break
					
				with torch.no_grad():
					action_dist = actor_net(state_tensor)
					sampled_action = action_dist.sample()
					state_value = critic_net(state_tensor)

					BatchLogProbs = action_dist.log_prob(sampled_action).sum(dim=-1)
				
				MemoryActions = []

				for i,con in enumerate(ConList):
					steer_val = torch.clamp(sampled_action[i, 0], min=-1.0, max=1.0).item()
					accel_val = torch.clamp(sampled_action[i, 1], min=0.0, max=1.0).item()
					MemoryActions.append((steer_val,accel_val))

					if BatchDones[i]:
						con.send('TERMINATE')
					else:
						con.send((steer_val,accel_val))

				NextStates = []
				PreviousRewards = []
				PreviousDones = []

				for con in ConList:
						np_obs,reward,RaceDone = con.recv()
						NextStates.append(np_obs)
						PreviousRewards.append(reward)
						PreviousDones.append(RaceDone)

				for i in range(len(ConList)):
					transition = {
						"state": BatchStates[i],
						"action": np.array([MemoryActions[i][1], MemoryActions[i][0], 0.0, 0.0, 0.0], dtype=np.float32),
						"reward": PreviousRewards[i],
						"value": state_value[i].item(),
						"log_prob": BatchLogProbs[i].item(),
						"done": PreviousDones[i]
					}
					# Buffer Appending
					buffer.append(transition)
				
				total_episode_reward += sum(PreviousRewards)

				BatchStates = NextStates
				BatchDones = PreviousDones
					
			# --- END OF EPISODE TRIGGER ---
			print(f"Episode: {episode + 1}/100 | Total Reward: {total_episode_reward:.2f} | Buffer Size: {len(buffer)}")

			# Trigger the Brain Transplant!
			update_ppo(buffer)

			mean_reward = total_episode_reward / max(len(buffer), 1)
			if mean_reward > best_reward:
				best_reward = mean_reward
				patience_counter = 0
				torch.save(actor_net.state_dict(), "best_actor.pth")
				torch.save(critic_net.state_dict(), "best_critic.pth")
				print(f"  New best reward: {best_reward:.4f} - models saved.")
			else:
				patience_counter += 1
				print(f"  No improvement. Patience: {patience_counter}/{PATIENCE_LIMIT}")

			if patience_counter >= PATIENCE_LIMIT:
				print(f"\nEarly stopping triggered after {episode + 1} episodes.")
				break

			for con in ConList:
				con.send('TERMINATE')

			for process in ProcessList:
				process.join()
	
	finally:
		for con in ConList:
			con.send('TERMINATE')

		for process in ProcessList:
			process.join()

In [6]:
if __name__ == '__main__':
	main()

..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
Episode: 1/100 | Total Reward: 402.42 | Buffer Size: 5000


/tmp/ipykernel_343372/3444157427.py:19: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  states = torch.FloatTensor([t['state'] for t in buffer])


  New best reward: 0.0805 - models saved.
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
Episode: 2/100 | Total Reward: 362.57 | Buffer Size: 5000
  No improvement. Patience: 1/50
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
Episode: 3/100 | Total Reward: 284.37 | Buffer Size: 5000
  No improvement. Patience: 2/50
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::....:: Antarctica Rendering Engine 2.0 ::..

..:: Antarctica Rendering Engine 2.0 ::..
Episode: 4/100 | Total Reward: 237.50 | Buffer Size: 5000
  No improvement. Patience: 3/50
..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarcti